# M4b — Making CP-SAT struggle

**2026 Quantum Accelerator · Day 1 · optional, ~25 minutes · bridge to M5**

In M4 you watched simulated annealing fall off a cliff while OR-Tools CP-SAT solved the identical
model in 17 milliseconds and proved it optimal. The fair conclusion from that module is
uncomfortable: *the roster problem was never hard — our encoding of it was.*

So the obvious next question, and the right one to ask before anyone sells you a solver:

> **What would actually make a constraint solver struggle?**

The answer is not "more variables" and it is not "more constraints". We will measure both and watch
them fail to matter. What does matter is the **kind** of constraint, and how close you are to the
boundary between possible and impossible.

---
## 0. What we already know does *not* work

Two negative results, measured on the M4 model before this notebook was written. Both are worth a
minute of your attention because they eliminate the intuitions most people arrive with.

**Adding variables barely helps.** Same six constraint families, moderate workload, scaling the
ward up:

| nurses × days | variables | CP-SAT |
|---|---|---|
| 12 × 14 | 168 | 0.017 s |
| 48 × 56 | 2,688 | 0.121 s |
| 200 × 224 | 44,800 | 2.35 s |
| 400 × 448 | 179,200 | unresolved at 15 s |

Six orders of magnitude of search space bought us two orders of magnitude of solve time. You need
about 10⁵ variables before size alone is the problem.

**Tightening the constraints helps even less.** Fixed ward of 24 × 28, pushing the workload from 8
days up to 14 — which is the absolute maximum the rest rule permits, where the only legal patterns
are perfectly alternating:

| days per nurse | 8 | 9 | 10 | 11 | 12 | 13 | 14 (max) |
|---|---|---|---|---|---|---|---|
| CP-SAT | 0.044 s | 0.051 s | 0.038 s | 0.042 s | 0.040 s | 0.049 s | **0.031 s** |

Nothing. Not even a wobble at the razor's edge.

**Why.** Coverage, workload, weekend quotas and sliding-window rest rules are all *counting*
constraints. Together they form something very close to a flow problem, and CP-SAT's propagators
cut straight through it. Stacking up more of them does not help, because they are all the same kind
of easy.

So we need a constraint of a different kind.

---
## 1. One new rule

**Some pairs of nurses must not be rostered on the same day.** Personality clashes, supervision
rules, a union agreement, two people who must not both be away from another ward — the reason does
not matter. The structure does.

This single rule changes the character of the problem. Each day's roster must now be an
**independent set** in a conflict graph: a set of nurses with no forbidden pair among them. Finding
independent sets of a required size is one of the classic hard problems, and unlike counting
constraints there is no flow structure for a solver to exploit.

We control one dial: the **conflict rate**, the fraction of all nurse pairs that are forbidden.

In [ ]:
import numpy as np
import time
import helpers
from ortools.sat.python import cp_model
from pyqubo import Array, Constraint, Placeholder
from dwave.samplers import SimulatedAnnealingSampler

N_NURSES, N_DAYS, DAYS_PER_NURSE = 36, 28, 9      # 1008 variables


def make_instance(conflict_rate, seed=1):
    """Roster requirement per day, plus the list of forbidden nurse pairs."""
    total = N_NURSES * DAYS_PER_NURSE
    base = total // N_DAYS
    required = np.full(N_DAYS, base)
    for i in range(total - base * N_DAYS):
        required[i] += 1                           # spread the remainder
    rng = np.random.default_rng(seed)
    pairs = [(a, b) for a in range(N_NURSES) for b in range(a + 1, N_NURSES)
             if rng.random() < conflict_rate]
    return required, pairs


required, pairs = make_instance(0.12, seed=1)
print(f"{N_NURSES} nurses x {N_DAYS} days = {N_NURSES * N_DAYS} variables")
print(f"nurses on duty per day : {required.tolist()}")
print(f"forbidden pairs        : {len(pairs)} of {N_NURSES * (N_NURSES - 1) // 2} possible")
print()
print("Each day must pick an independent set of the required size from this graph:")
helpers.show_conflict_graph(N_NURSES, pairs)

---
## 2. The two models

Both get exactly the same rules: coverage, workload, no two consecutive days, and now the forbidden
pairs. CP-SAT states them as constraints; the QUBO states them as costs.

In [ ]:
def solve_cpsat(conflict_rate, seed=1, time_limit=5.0):
    required, pairs = make_instance(conflict_rate, seed)
    m = cp_model.CpModel()
    x = {(n, d): m.NewBoolVar("") for n in range(N_NURSES) for d in range(N_DAYS)}
    for d in range(N_DAYS):
        m.Add(sum(x[n, d] for n in range(N_NURSES)) == int(required[d]))
    for n in range(N_NURSES):
        m.Add(sum(x[n, d] for d in range(N_DAYS)) == DAYS_PER_NURSE)
        for d in range(N_DAYS - 1):
            m.Add(x[n, d] + x[n, d + 1] <= 1)
    for a, b in pairs:                                     # the new rule
        for d in range(N_DAYS):
            m.Add(x[a, d] + x[b, d] <= 1)
    s = cp_model.CpSolver()
    s.parameters.max_time_in_seconds = float(time_limit)
    s.parameters.num_workers = 8
    status = s.Solve(m)
    return s.StatusName(status), s.WallTime()


SAMPLER = SimulatedAnnealingSampler()

def solve_annealing(conflict_rate, seed=1, num_reads=40, num_sweeps=4000):
    required, pairs = make_instance(conflict_rate, seed)
    x = Array.create("x", shape=(N_NURSES, N_DAYS), vartype="BINARY")
    w = {f: Placeholder(f"w_{f}") for f in ("coverage", "workload", "rest", "conflict")}
    H = 0
    for d in range(N_DAYS):
        H += w["coverage"] * Constraint(
            (sum(x[n, d] for n in range(N_NURSES)) - int(required[d]))**2, label=f"cov_d{d:02d}")
    for n in range(N_NURSES):
        H += w["workload"] * Constraint(
            (sum(x[n, d] for d in range(N_DAYS)) - DAYS_PER_NURSE)**2, label=f"load_n{n:02d}")
        for d in range(N_DAYS - 1):
            H += w["rest"] * Constraint(x[n, d] * x[n, d + 1], label=f"rest_n{n:02d}_d{d:02d}")
    for a, b in pairs:                                     # the same new rule, as a cost
        for d in range(N_DAYS):
            H += w["conflict"] * Constraint(x[a, d] * x[b, d], label=f"conf_{a}_{b}_d{d:02d}")

    model = H.compile()
    wd = {"w_coverage": 2.0, "w_workload": 2.0, "w_rest": 5.0, "w_conflict": 5.0}
    ss = SAMPLER.sample(model.to_bqm(feed_dict=wd), num_reads=num_reads,
                        num_sweeps=num_sweeps, seed=1)
    decoded = model.decode_sampleset(ss, feed_dict=wd)
    broken = np.array([len(d.constraints(only_broken=True)) for d in decoded])
    return 100 * float((broken == 0).mean()), int(broken.min()), len(decoded[0].constraints())

---
## 3. Exercise — find the wall

**You type one list.** Sweep the conflict rate. Three different random conflict graphs per rate, so
you can see the spread as well as the trend. Roughly half a minute.

In [ ]:
CONFLICT_RATES = [0.06, 0.08, 0.10, 0.12, 0.14]      # <-- CHANGE ME
TIME_LIMIT = 5.0
SEEDS = [1, 2, 3]

times, statuses = [], []
for rate in CONFLICT_RATES:
    row_t, row_s = [], []
    for sd in SEEDS:
        st, wall = solve_cpsat(rate, seed=sd, time_limit=TIME_LIMIT)
        row_t.append(min(wall, TIME_LIMIT))
        row_s.append(st)
    times.append(row_t)
    statuses.append(row_s)
    print(f"  conflict rate {rate:4.2f}   " +
          "   ".join(f"{s[:4]}/{t:5.2f}s" for s, t in zip(row_s, row_t)))

helpers.plot_transition(CONFLICT_RATES, times, TIME_LIMIT)

### Two things to notice

**The wall is razor-sharp.** Between a conflict rate of 0.10 and 0.12 — about **nine extra
forbidden pairs out of 630** — CP-SAT goes from answering in a tenth of a second to not answering
at all. Nothing about the size of the problem changed. One dial moved by two percentage points.

**Inside the wall, identical parameters give wildly different instances.** At the same conflict
rate, one graph is solved in about 0.1 s and its neighbour is still running when the clock runs out.
Given a longer leash — 25 seconds instead of 5 — three of four instances at a rate of 0.12 were
still unresolved while the fourth finished in 0.13 s. That is a spread of more than 200× between
instances generated from the same parameters.

That is the signature of a **phase transition**. Well below the boundary, solutions are plentiful
and any solver trips over one. Well above it, infeasibility is easy to prove by a counting
argument. Only *at* the boundary does the search have to do real work — and there, difficulty stops
being a smooth function of anything you can measure in advance.

This is the single most useful thing to take back to your own work. "How many variables does it
have?" is the wrong question. "How close is it to the edge of feasible?" is the right one, and it is
much harder to answer.

---
## 4. And the annealer?

Fair is fair. The same four rates, the same instances, through the QUBO.

In [ ]:
sa_feas = []
for rate in CONFLICT_RATES:
    t0 = time.perf_counter()
    feas, fewest, n_cons = solve_annealing(rate, seed=1)
    sa_feas.append(feas)
    cp_status, cp_wall = solve_cpsat(rate, seed=1, time_limit=TIME_LIMIT)
    print(f"  rate {rate:4.2f} ({n_cons:>4} constraints) | "
          f"annealing: {feas:5.1f}% feasible, best broke {fewest:>2} | "
          f"CP-SAT: {cp_status:<11} {cp_wall:5.2f}s   [{time.perf_counter() - t0:.0f}s]")

helpers.plot_transition(CONFLICT_RATES, times, TIME_LIMIT, sa_feasible=sa_feas)

### Three regimes, and only one of them is interesting

| conflict rate | annealing | CP-SAT | what it means |
|---|---|---|---|
| ≤ 0.06 | finds a roster every run | instant | **both easy** — plenty of slack |
| ~0.10 | never finds a roster | instant | **the encoding tax** — M4's lesson, at 1,008 variables |
| ≥ 0.12 | never finds a roster | never finishes | **the frontier** — nobody has a good answer |

Note carefully what did *not* happen: there is no rate at which the annealer beats CP-SAT. The
honest picture is that the QUBO encoding gives up first, and then — a little further along — the
constraint solver gives up too. Anyone claiming a quantum or quantum-inspired advantage owes you a
plot like this one, with the classical baseline in the same notebook.

What the third regime *is* good for is knowing where the research frontier actually lies. Problems
in that band are where new methods have something to prove, and they are exactly where the
community has pointed its benchmark suite.

---
## 5. Which is why the next module looks the way it does

After the break we move to **sports tournament scheduling** from QOBLIB, the Quantum Optimization
Benchmarking Library assembled by the Quantum Optimization Working Group that IBM Quantum and
partners started in 2023, published in *Nature Computational Science* in 2026.

Round-robin tournament scheduling has both ingredients you just met. It has counting constraints —
each team plays once per slot, so many home games in a window. And it has an irreducibly
graph-structured core: *who plays whom*, home/away balance, minimum separation between repeat
fixtures. You cannot factor that into a flow problem.

Then the maintainers did deliberately what you just did by accident. They stripped the soft
constraints to leave pure feasibility, and selected the instances **"for which existing solvers
could not find any feasible solution within a reasonable amount of time"** — at sixteen to twenty
teams, a little over a hundred variables.

A hundred variables. You have just spent twenty minutes establishing that variable count is nearly
irrelevant and that the interesting instances live in a narrow band at the feasibility boundary.
QOBLIB is a library of nothing but that band.

---
## 6. Scope, honestly

Two limits on what we just showed, so you can weigh it properly.

**The counting-constraint result is specific to this model.** Real nurse rostering has shift *types*
— day, evening, night — which triples the variables and adds forbidden shift-sequence rules that
are genuinely awkward for a solver. Published instances from the International Nurse Rostering
Competition are harder than ours. Our conclusion is "*this* roster model is easy for CP-SAT", not
"rostering is easy".

**"Unresolved in five seconds" is not "unsolvable".** Some of those instances would fall to CP-SAT
given minutes, or to a solver with a better cut for independent sets, or to a metaheuristic tuned to
the structure. The claim is about a workshop time budget, not about complexity theory. Keep that
distinction when you repeat any of this to a sceptic — it is the difference between a credible
result and a vendor slide.